In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date
import os
import warnings

warnings.filterwarnings("ignore")

def normalize_col(col):
    if isinstance(col, str):
        return " ".join(col.strip().lower().replace("\n", " ").split())
    return col

def is_date_col(col):
    return isinstance(col, (pd.Timestamp, datetime, np.datetime64, date))

first_glob = os.path.expanduser("~").replace("\\", "/")
file_path = os.path.join(
    first_glob,
    "Concentrix Corporation",
    "WFM-Expedia-HCM - Branding files",
    "BI_Task",
    "CODE",
    "Operations HC file.xlsm"
)

sheet_names = [
    "Kolkata", "Pune", "Cairo", "Vietnam", "Latvia", "Lithuania", "China",
    "Marrakech", "Tirana", "Halifax", "HSB", "Malaga", "Surinam",
    "Estonia", "Amsterdam", "LCR", "Gurugram"
]

cols_to_keep = [
    "IEX ID", "Emp ID", "Name", "Supervisor", "SSO ID",
    "Chat/Voice/Blended", "Mini Team TL", "WAH/B&M", "Hire Date",
    "Production Start date", "Termination Date", "Agent/Non Agent"
]

cols_to_keep_norm = [normalize_col(col) for col in cols_to_keep]
all_dfs = []

for sheet in sheet_names:
    try:
        df_sheet = pd.read_excel(
            file_path,
            sheet_name=sheet,
            header=0,
            engine="calamine", 
            na_values=["-"]
        )

        columns_orig = list(df_sheet.columns)
        columns_norm = [normalize_col(col) for col in columns_orig]

        if "agent/ non agent" in columns_norm:
            last_info_col_idx = columns_norm.index("agent/ non agent")
        elif "agent/non agent" in columns_norm:
            last_info_col_idx = columns_norm.index("agent/non agent")
        else:
            print(f"Skipped {sheet} - Not found column 'Agent/Non Agent'")
            continue

        col_map = {
            normalize_col(orig): orig
            for orig in columns_orig
            if isinstance(orig, str)
        }

        info_cols = [col_map[col] for col in cols_to_keep_norm if col in col_map]
        missing_cols = [col for col in cols_to_keep_norm if col not in col_map]

        if missing_cols:
            print(f"⚠️ {sheet} - Missing columns: {missing_cols}")

        week_cols = []
        i = last_info_col_idx + 1
        while i < len(columns_orig):
            col = columns_orig[i]
            if is_date_col(col):
                week_cols.append(col)
                i += 1
            else:
                break

        use_cols = info_cols + week_cols
        temp = df_sheet[use_cols].copy()
        temp["Location"] = sheet

        print(f"✅ {sheet} - Number of week columns found: {len(week_cols)}")
        all_dfs.append(temp)

    except Exception as e:
        print(f"❌ Error reading sheet {sheet}: {e}")

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print("\n🚀 Final dataframe shape:", df.shape)
else:
    df = pd.DataFrame()
    print("\n⚠️ No sheets found.")

⚠️ Kolkata - Missing columns: ['chat/voice/blended']
✅ Kolkata - Number of week columns found: 62
✅ Pune - Number of week columns found: 62
⚠️ Cairo - Missing columns: ['hire date']
✅ Cairo - Number of week columns found: 62
⚠️ Vietnam - Missing columns: ['chat/voice/blended']
✅ Vietnam - Number of week columns found: 62
⚠️ Latvia - Missing columns: ['hire date']
✅ Latvia - Number of week columns found: 62
⚠️ Lithuania - Missing columns: ['hire date']
✅ Lithuania - Number of week columns found: 62
⚠️ China - Missing columns: ['hire date']
✅ China - Number of week columns found: 0
⚠️ Marrakech - Missing columns: ['hire date']
✅ Marrakech - Number of week columns found: 62
⚠️ Tirana - Missing columns: ['hire date']
✅ Tirana - Number of week columns found: 0
⚠️ Halifax - Missing columns: ['hire date']
✅ Halifax - Number of week columns found: 62
⚠️ HSB - Missing columns: ['hire date']
✅ HSB - Number of week columns found: 62
⚠️ Malaga - Missing columns: ['hire date']
✅ Malaga - Number of 

In [20]:
df.columns = [col.strftime('%#d-%b-%y') if isinstance(col, datetime) else col for col in df.columns]

df = df.dropna(subset=["SSO ID"]).copy()

for col in ["Production Start date", "Hire Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

today = pd.Timestamp.today().normalize()
df["AON_Days"] = (today - df["Production Start date"]).dt.days

conditions = [
    df["AON_Days"].isna(),
    df["AON_Days"] > 180,
    df["AON_Days"] >= 91,
    df["AON_Days"] >= 61,
    df["AON_Days"] >= 31,
    df["AON_Days"] >= 0
]
choices = [
    "Nesting", "> 180 Days", "91 - 180", "61 - 90", "31 - 60", "00 - 30"
]

aon_status_array = np.select(conditions, choices, default="Future")
df["AON Status"] = np.where(df["Agent/Non Agent"] == "Agent", aon_status_array, None)

df = df.drop(columns=["AON_Days"])

id_cols = [
    'IEX ID', 'Emp ID', 'Name', 'Supervisor', 'SSO ID',
    'Mini Team TL', 'WAH/B&M', 'Hire Date', 'Production Start date',
    'Termination Date', 'Agent/Non Agent', 'Location',
    'Chat/Voice/Blended', 'AON Status'
]

week_cols = [col for col in df.columns if col not in id_cols]

df_long = pd.melt(
    df,
    id_vars       = id_cols,
    value_vars    = week_cols,
    var_name      = 'Week_String',
    value_name    = 'LOB'
)

parsed_dates = pd.to_datetime(df_long['Week_String'], errors='coerce')
df_long['Week_Monday'] = parsed_dates - pd.to_timedelta(parsed_dates.dt.weekday, unit='D')

df_long = df_long.drop(columns=['Week_String'])

desired_order = id_cols + ['Week_Monday', 'LOB']
df_long = df_long[desired_order]
df_long = df_long.sort_values(['Emp ID', 'Week_Monday'])

output_dir = os.path.join(first_glob, "Concentrix Corporation", "WFM-Expedia-HCM - Branding files", "BI_Task", "CODE")
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "Global_HC.csv")
df_long.to_csv(output_file, index=False)

object_columns = df_long.select_dtypes(include=['object']).columns
for col in object_columns:
    df_long[col] = df_long[col].astype('string')

df_long.to_parquet(output_file.replace('.csv', '.parquet'), index=False)